# Phase 6 — Full Audit of All Methods
Merges all results and computes Wilcoxon tests.

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path
from scipy.stats import wilcoxon
print('Imports successful')

Imports successful


In [2]:
BASE    = Path('E:/project1/EW_MOGA_Project')
RESULTS = BASE / 'results'
print('Configuration set')

Configuration set


In [3]:
# ── Load all result files ─────────────────────────────────────
files = {
    'EW-MOGA'         : RESULTS / 'ew_moga_results.csv',
    'Baselines'       : RESULTS / 'baselines_results.csv',
    'New Baselines'   : RESULTS / 'new_baselines_results.csv',
}

dfs = []
for name, path in files.items():
    if path.exists():
        df = pd.read_csv(path)
        dfs.append(df)
        print(f'Loaded {name}: {df.shape} — methods: {df.method.unique().tolist()}')
    else:
        print(f'WARNING: {path} not found — run the corresponding notebook first')

df_all = pd.concat(dfs, ignore_index=True)
print(f'\nTotal rows: {df_all.shape}')
print(f'All methods: {df_all.method.unique().tolist()}')

Loaded EW-MOGA: (15, 10) — methods: ['EW_MOGA']
Loaded Baselines: (45, 10) — methods: ['Standard_GA', 'NSGA2_Parsimony', 'SHAP_RF']
Loaded New Baselines: (45, 10) — methods: ['Boruta', 'RFE', 'mRMR']

Total rows: (105, 10)
All methods: ['EW_MOGA', 'Standard_GA', 'NSGA2_Parsimony', 'SHAP_RF', 'Boruta', 'RFE', 'mRMR']


In [4]:
# ── Summary statistics ───────────────────────────────────────
summary = df_all.groupby('method').agg(
    accuracy_mean =('accuracy',   'mean'),
    accuracy_std  =('accuracy',   'std'),
    n_feat_mean   =('n_features', 'mean'),
    n_feat_std    =('n_features', 'std'),
    runtime_mean  =('runtime_sec','mean'),
    ccc_mean      =('audit_ccc',  'mean'),
    ccc_std       =('audit_ccc',  'std'),
    pr_mean       =('audit_pr',   'mean'),
    pr_std        =('audit_pr',   'std'),
    n_runs        =('accuracy',   'count'),
).round(4)

summary.to_csv(RESULTS / 'summary_statistics.csv')
print('Summary statistics:')
print(summary.to_string())
print('\nSaved summary_statistics.csv')

Summary statistics:
                 accuracy_mean  accuracy_std  n_feat_mean  n_feat_std  runtime_mean  ccc_mean  ccc_std  pr_mean  pr_std  n_runs
method                                                                                                                         
Boruta                  0.9872        0.0038      12.0000      0.0000       77.5747    0.4120   0.0521   0.9438  0.0203      15
EW_MOGA                 0.9902        0.0035       5.7333      1.3345      742.7916    0.6210   0.1451   0.9651  0.0324      15
NSGA2_Parsimony         0.9907        0.0035       4.8667      1.4075      296.1950    0.3382   0.3272   0.9305  0.1011      15
RFE                     0.9845        0.0025       6.0000      0.0000        9.1513    0.5092   0.0939   0.9208  0.0222      15
SHAP_RF                 0.8351        0.0123       6.0000      0.0000        4.7976    0.0660   0.2008   0.1383  0.3462      15
Standard_GA             0.9929        0.0031       7.4667      1.6417      474.8103 

In [5]:
# ── Wilcoxon signed-rank tests ────────────────────────────────
def paired_wilcoxon(df, metric, method_a, method_b):
    pivot = df[df['method'].isin([method_a, method_b])].pivot_table(
        index=['seed','fold'], columns='method', values=metric)
    if method_a not in pivot.columns or method_b not in pivot.columns:
        return None, None, 0
    valid = pivot[[method_a, method_b]].dropna()
    n     = len(valid)
    if n < 6:
        return None, None, n
    try:
        stat, p = wilcoxon(valid[method_a].values, valid[method_b].values)
        return float(stat), float(p), n
    except:
        return None, None, n

EW_MOGA = 'EW_MOGA'
baselines = [m for m in df_all['method'].unique() if m != EW_MOGA]
metrics   = ['accuracy', 'audit_ccc', 'audit_pr']

wilcoxon_results = []
for metric in metrics:
    for baseline in baselines:
        stat, p, n = paired_wilcoxon(df_all, metric, EW_MOGA, baseline)
        wilcoxon_results.append({
            'metric'     : metric,
            'baseline'   : baseline,
            'W_statistic': stat,
            'p_value'    : p,
            'n_pairs'    : n,
            'significant': bool(p < 0.05) if p is not None else None,
        })

df_wilcoxon = pd.DataFrame(wilcoxon_results)
df_wilcoxon.to_csv(RESULTS / 'wilcoxon_results.csv', index=False)
print('Wilcoxon results:')
print(df_wilcoxon.to_string(index=False))
print('\nSaved wilcoxon_results.csv')

Wilcoxon results:
   metric        baseline  W_statistic  p_value  n_pairs  significant
 accuracy     Standard_GA         29.5 0.083161       15        False
 accuracy NSGA2_Parsimony         50.0 0.875200       15        False
 accuracy         SHAP_RF          0.0 0.000653       15         True
 accuracy          Boruta          0.0 0.002174       15         True
 accuracy             RFE          0.0 0.000648       15         True
 accuracy            mRMR          0.0 0.000647       15         True
audit_ccc     Standard_GA          1.0 0.000122       15         True
audit_ccc NSGA2_Parsimony         22.0 0.030151       15         True
audit_ccc         SHAP_RF          1.0 0.000122       15         True
audit_ccc          Boruta          2.0 0.000183       15         True
audit_ccc             RFE         20.0 0.023068       15         True
audit_ccc            mRMR         48.0 0.495434       15        False
 audit_pr     Standard_GA          1.0 0.000122       15         True
 a

In [6]:
# ── Save merged results ──────────────────────────────────────
df_all.to_csv(RESULTS / 'all_results_merged.csv', index=False)
print(f'Saved all_results_merged.csv ({df_all.shape[0]} rows)')
print()
print('='*55)
print('Notebook 6 complete — ready to run Notebook 7')
print('='*55)

Saved all_results_merged.csv (105 rows)

Notebook 6 complete — ready to run Notebook 7
